# 01 - Gradient Practice

这本 `course` 不是压缩版提纲。课堂 notebook 会先把直觉、例子、代码和图跑通，再进入最后的 Predict / Modify 检查。

学习顺序：先读这一页的主线，遇到代码就运行；最后再做底部的小检查。真正写作业时进入同目录的 `homework.ipynb`。

In [ ]:
from pathlib import Path
import sys, math, random, inspect


def find_repo_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / 'micrograd' / 'engine.py').exists():
            return q
    return p

ROOT = find_repo_root()
PROJECT_ROOT = ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ModuleNotFoundError:
    plt = None
    MATPLOTLIB_AVAILABLE = False

try:
    import torch
except ModuleNotFoundError:
    torch = None


def near(a, b, eps=1e-6):
    return abs(a - b) < eps


def assert_close(actual, expected, name='value', eps=1e-9):
    assert abs(actual - expected) < eps, f'{name}: expected {expected}, got {actual}'


def todo_guard(values, message='请先填写 TODO，再运行检查。'):
    if any(v is None for v in values):
        print(message)
        return False
    return True

import importlib
import course.checks as course_checks
importlib.reload(course_checks)
qa_check = course_checks.qa_check


def show_svg(path):
    path = Path(path)
    try:
        from IPython.display import SVG, display
        display(SVG(filename=str(path)))
    except Exception:
        print('图已生成，但当前环境不能内嵌显示。请打开:', path.resolve())

print('project root:', ROOT)


## 为什么要练手算

这一节只抓一句话：

> 先会手算小表达式的梯度，后面看 `backward()` 才不会像黑盒。

直觉上，micrograd 做的事并不神秘：

```text
每个小运算只知道自己的局部变化率；
反向传播把这些局部变化率沿路径乘起来；
如果同一个变量有多条路径影响 L，就把贡献加起来。
```

所以这本不是数学考试，而是在给源码理解铺路。

## 怎么练这本

这本不要一口气全跑完。正确方式是：

1. 先读题目，停下来手算
2. 写下你认为的 `a.grad` 和 `b.grad`
3. 再运行代码 cell 校验
4. 最后点击 `Show / Hide 答案` 展开答案

如果你直接一路 Run All，会失去练习效果。

第一轮只做 1-5 题；第二轮再做 6-10 题。


## 链式法则速记

遇到复杂一点的式子，先拆中间变量：

```text
x -> u -> y
```

就记：

$$
\frac{dy}{dx}=\frac{dy}{du}\cdot\frac{du}{dx}
$$

如果路径更长：

```text
x -> u -> v -> L
```

就继续乘：

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial v}
\cdot
\frac{\partial v}{\partial u}
\cdot
\frac{\partial u}{\partial x}
$$

如果同一个变量通过多条路径影响 $L$，就把每条路径的贡献相加。


## 手算模板

每题都按这个模板：

```text
1. 写出完整公式
2. 如果公式简单，直接求偏导
3. 如果公式有中间变量，画成几步：u=..., v=..., L=...
4. 使用链式法则：局部变化率相乘
5. 如果一个变量有多条路径影响 L，把路径贡献相加
```

小公式优先直接求偏导；路径法主要用来理解 micrograd 自动求导。


## 作业模式：先写你的答案

每道题下面都有一个留空 cell：

```python
your_a_grad = None
your_b_grad = None
qa_check('ex1', your_a_grad, your_b_grad)
```

做题时先把 `None` 改成你手算的答案，再运行 cell。

```text
如果还没填，qa_check 会提示 TODO。
如果填错，qa_check 会指出 expected / got。
如果填对，qa_check 会打印 OK。
```

提示和答案是分开的：先看 `Show / Hide 提示`，最后再看 `Show / Hide 答案`。

## 先把一道题拆成路径贡献

手算梯度不是背答案，而是把 loss 拆成路径。

以 `L = 2(ab + a)` 为例，`a=2, b=3`。

`a` 到 `L` 有两条路：

```text
a -> ab -> ab+a -> 2(ab+a)
a -> a  -> ab+a -> 2(ab+a)
```

`b` 到 `L` 只有一条路：

```text
b -> ab -> ab+a -> 2(ab+a)
```

所以 `a.grad` 要把两条贡献加起来；`b.grad` 只收一条贡献。

In [ ]:
a = 2
b = 3

path_table = [
    ('a -> ab -> d -> L', 'b', 2, b * 2),
    ('a -> a  -> d -> L', '1', 2, 1 * 2),
    ('b -> ab -> d -> L', 'a', 2, a * 2),
]

print('path | local derivative | upstream | contribution')
for path, local, upstream, contribution in path_table:
    print(f'{path:20s} | {local:>5s} | {upstream:>8} | {contribution:>12}')

print('dL/da =', b * 2 + 1 * 2)
print('dL/db =', a * 2)

## 练习时怎么避免乱

每一道题都按同一张表走：

```text
1. 写出 forward 公式。
2. 选一个变量，比如 a。
3. 找出 a 到 L 的所有路径。
4. 每条路径：局部导数一路相乘。
5. 多条路径：贡献相加。
```

你之前做复杂图时已经摸到这个点了：难的不是公式，是细心地不要漏路径。

In [ ]:
def explain_gradient_case(name, paths):
    print(name)
    total = 0
    for path, contribution in paths:
        print(' ', path, '=>', contribution)
        total += contribution
    print(' total grad =', total)
    print()

explain_gradient_case('L = a + a, a=2', [
    ('left a -> L', 1),
    ('right a -> L', 1),
])

explain_gradient_case('L = (2w+1)^2, w=3', [
    ('w -> 2w+1 -> square', 2 * (2 * 3 + 1) * 2),
])

## Debug Lab - `+=` 为什么是保命符

如果一个变量有多条路到 loss，反传时每条路都会贡献一份 grad。

```text
L = a + a
第一条路给 a.grad 贡献 1
第二条路又给 a.grad 贡献 1
最后 a.grad = 2
```

如果你在实现里写 `a.grad = ...`，第二条路会覆盖第一条路。
如果写 `a.grad += ...`，两条路才会加起来。

## What To Remember

- 跑通主线。
- 说清楚本节的输入、输出、梯度或训练含义。
- 完成底部课堂检查后再进入 homework。

## 课堂检查：先预测，再改一行

这一段保留 `course` 的隐藏检查。你应该先自己填，再展开提示或答案。

## Predict - 乘法局部梯度

In [ ]:
# 填写说明：
# - 填两个数字：L=a*b 时 a、b 各自收到的局部梯度。
# - 填完后运行本 cell，看 qa_check 的提示。

# L=a*b, a=2,b=3
predict_dL_da = None
predict_dL_db = None



qa_check('qa_check_class_01_predict', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先写局部导数：加法传 1，乘法乘另一个输入；如果同一个变量出现两次，要把两条贡献相加。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `predict_dL_da` 填 `3`。
- `predict_dL_db` 填 `2`。

</details>

## Run - 数值估计 dL/da

In [ ]:
a = 2.0
b = 3.0
h = 0.0001
L1 = a*b
L2 = (a+h)*b
print('numeric dL/da =', (L2-L1)/h)

## 拆开看 - 重复变量有两条路径

In [ ]:
a = 3
print('L = a*a')
print('path 1 contribution:', a)
print('path 2 contribution:', a)
print('dL/da:', 2*a)

## Modify - 把 `a*b` 改成 `a*b+a`

In [ ]:
# 填写说明：
# - 填两个数字：L=a*b+a 时 dL/da 和 dL/db。
# - 填完后运行本 cell，看 qa_check 的提示。

# a=2,b=3, L=a*b+a
student_dL_da = None
student_dL_db = None



qa_check('qa_check_class_01_modify', globals())

<details><summary>Show / Hide 本题提示</summary>

- 先写局部导数：加法传 1，乘法乘另一个输入；如果同一个变量出现两次，要把两条贡献相加。

</details>

<details><summary>Show / Hide 本题答案</summary>


参考答案（先自己做，卡住再展开）：

- `student_dL_da` 填 `4`。
- `student_dL_db` 填 `2`。

</details>